LIBRARIES

In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

/home/nineleaps/Documents/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KNOWLEDGE BASE

In [2]:
knowledge_base = [
    "To reset your password, click on the Forgot Password link on the login page.",
    "Billing invoices can be downloaded from the billing section of your account.",
    "If your account is locked, contact customer support for assistance.",
    "You can update your profile information from the account settings page.",
    "Subscription plans can be upgraded or downgraded anytime.",
    "If you cannot log in, ensure your username and password are correct.",
    "Refund requests are processed within five to seven business days.",
    "Two-factor authentication adds an extra layer of account security.",
    "Payment methods can be managed in the billing dashboard.",
    "Account deletion requests must be submitted through the support portal."
]

LOADING EMBEDDING MODEL

In [3]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5387.91it/s]


GENERATING EMBEDDINGS

In [4]:
embeddings = model.encode(knowledge_base)

print("\nEmbedding Matrix Shape:")
print(embeddings.shape)


Embedding Matrix Shape:
(10, 384)


BUILDING FAISS INDEX

In [5]:
dimension = embeddings.shape[1] 

index = faiss.IndexFlatL2(dimension)

NORMALIZING EMBEDDINGS FOR COSINE SIMILARITY BEHAVIOUR

In [6]:
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

ADDING VECTORS TO THE FAISS INDEX

In [7]:
index.add(embeddings)

print("\nTotal Vectors Stored in FAISS:")
print(index.ntotal)


Total Vectors Stored in FAISS:
10


SEMANTIC SEARCH FUNCTION

In [8]:
def semantic_search(query, k=3):
    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    print("\nTop Matches:")
    print("-" * 90)
    print(f"{'Rank':<6}{'Score':<15}{'Matched Sentence'}")
    print("-" * 90)

    for rank, (idx, score) in enumerate(
        zip(indices[0], distances[0]), start=1
    ):
        print(
            f"{rank:<6}{score:<15.4f}{knowledge_base[idx]}"
        )



TESTING QUERIES

In [9]:
test_queries = [
    "How do I change my password?",
    "Where can I see my invoice?",
    "I am unable to access my account"
]

print("\n\n========== SAMPLE SEARCH RESULTS ==========")

for query in test_queries:
    print(f"\nQuery: {query}")
    semantic_search(query)




========== SAMPLE SEARCH RESULTS ==========

Query: How do I change my password?

Top Matches:
------------------------------------------------------------------------------------------
Rank  Score          Matched Sentence
------------------------------------------------------------------------------------------
1     0.6840         To reset your password, click on the Forgot Password link on the login page.
2     0.9730         If your account is locked, contact customer support for assistance.
3     1.1529         You can update your profile information from the account settings page.

Query: Where can I see my invoice?

Top Matches:
------------------------------------------------------------------------------------------
Rank  Score          Matched Sentence
------------------------------------------------------------------------------------------
1     0.5805         Billing invoices can be downloaded from the billing section of your account.
2     1.2133         Payment method

INTERACTIVE CLI

In [10]:
print("\n\n========== INTERACTIVE SEARCH ==========")

while True:
    query = input("\nEnter your query (or type 'exit'): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    semantic_search(query)



========== INTERACTIVE SEARCH ==========

Top Matches:
------------------------------------------------------------------------------------------
Rank  Score          Matched Sentence
------------------------------------------------------------------------------------------
1     0.4376         To reset your password, click on the Forgot Password link on the login page.
2     0.8521         If your account is locked, contact customer support for assistance.
3     1.1907         If you cannot log in, ensure your username and password are correct.
Goodbye!
